# Benchmark Result Statistics

This notebook calculates summary statistics for RV benchmark sessions.

For both **baseline** and **abnormal** datasets, it computes:
- `elapsedMs` stats (count, mean, stddev, median, min/max, p95, p99)
- `rowsPerSec` stats (count, mean, stddev, median, min/max, p95, p99)
- Overhead percentage from average elapsed time.

In [1]:
from __future__ import annotations

import csv
import json
import math
from pathlib import Path
from statistics import mean, median, stdev


def percentile(values: list[float], p: float) -> float:
    if not values:
        return float('nan')
    if len(values) == 1:
        return values[0]
    s = sorted(values)
    rank = (p / 100.0) * (len(s) - 1)
    lower = int(math.floor(rank))
    upper = int(math.ceil(rank))
    if lower == upper:
        return s[lower]
    w = rank - lower
    return s[lower] + (s[upper] - s[lower]) * w


def describe(values: list[float]) -> dict[str, float]:
    if not values:
        return {
            'count': 0,
            'mean': float('nan'),
            'stddev': float('nan'),
            'median': float('nan'),
            'min': float('nan'),
            'max': float('nan'),
            'p95': float('nan'),
            'p99': float('nan'),
        }
    return {
        'count': len(values),
        'mean': mean(values),
        'stddev': stdev(values) if len(values) > 1 else 0.0,
        'median': median(values),
        'min': min(values),
        'max': max(values),
        'p95': percentile(values, 95.0),
        'p99': percentile(values, 99.0),
    }


def load_runs(csv_path: Path) -> dict[str, list[float]]:
    on_ms, off_ms, on_rps, off_rps = [], [], [], []
    with csv_path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            on_ms.append(float(row['onElapsedMs']))
            off_ms.append(float(row['offElapsedMs']))
            on_rps.append(float(row['onRowsPerSec']))
            off_rps.append(float(row['offRowsPerSec']))
    return {
        'onElapsedMs': on_ms,
        'offElapsedMs': off_ms,
        'onRowsPerSec': on_rps,
        'offRowsPerSec': off_rps,
    }


def analyze_mode(mode_name: str, csv_path: Path) -> dict:
    runs = load_runs(csv_path)
    on_ms_stats = describe(runs['onElapsedMs'])
    off_ms_stats = describe(runs['offElapsedMs'])
    on_rps_stats = describe(runs['onRowsPerSec'])
    off_rps_stats = describe(runs['offRowsPerSec'])

    overhead_pct = ((on_ms_stats['mean'] - off_ms_stats['mean']) / off_ms_stats['mean']) * 100.0

    return {
        'mode': mode_name,
        'sourceCsv': str(csv_path),
        'rvOn': {'elapsedMs': on_ms_stats, 'rowsPerSec': on_rps_stats},
        'rvOff': {'elapsedMs': off_ms_stats, 'rowsPerSec': off_rps_stats},
        'overheadPct': overhead_pct,
    }


def latest_session_dir(runs_dir: Path) -> Path:
    sessions = sorted([p for p in runs_dir.glob('session-*') if p.is_dir()])
    if not sessions:
        raise FileNotFoundError(f'No session-* folders found in: {runs_dir}')
    return sessions[-1]


def format_block(title: str, stats: dict[str, float], unit: str = '') -> str:
    return (
        f"{title}\n"
        f"  mean={stats['mean']:.4f}{unit} stddev={stats['stddev']:.4f}{unit} median={stats['median']:.4f}{unit}\n"
        f"  min={stats['min']:.4f}{unit} max={stats['max']:.4f}{unit} p95={stats['p95']:.4f}{unit} p99={stats['p99']:.4f}{unit}"
    )

In [2]:
# Configuration
# Optional override: set to a specific folder name such as 'session-20260506-100232'
FORCE_SESSION = None


def resolve_runs_dir(start: Path) -> Path:
    """Find runs directory even if notebook is launched elsewhere."""
    candidates = [
        start / 'runs',
        start / 'src' / 'resultCollection' / 'runs',
    ]
    for parent in [start] + list(start.parents):
        candidates.append(parent / 'runs')
        candidates.append(parent / 'src' / 'resultCollection' / 'runs')

    for c in candidates:
        if c.exists() and c.is_dir() and any(c.glob('session-*')):
            return c
    raise FileNotFoundError(
        f'Could not locate a runs directory from start path: {start}. '
        'Expected runs/session-* or src/resultCollection/runs/session-*.'
    )


start_dir = Path.cwd()
runs_dir = resolve_runs_dir(start_dir)

if FORCE_SESSION:
    session_dir = runs_dir / FORCE_SESSION
else:
    session_dir = latest_session_dir(runs_dir)

baseline_csv = session_dir / 'baseline' / 'benchmark-runs.csv'
abnormal_csv = session_dir / 'abnormal' / 'benchmark-runs.csv'

if not baseline_csv.exists() or not abnormal_csv.exists():
    raise FileNotFoundError('Missing benchmark-runs.csv in baseline/abnormal folders for selected session.')

baseline = analyze_mode('baseline', baseline_csv)
abnormal = analyze_mode('abnormal', abnormal_csv)

report = {
    'sessionDir': str(session_dir),
    'baseline': baseline,
    'abnormal': abnormal,
}

print(f'Runs directory: {runs_dir}')
print(f'Session: {session_dir}')
print('Computed statistics for baseline and abnormal.')

Runs directory: C:\Users\brama\Documents\RD2\src\resultCollection\runs
Session: C:\Users\brama\Documents\RD2\src\resultCollection\runs\session-20260509-105518
Computed statistics for baseline and abnormal.


In [3]:
# Show results as tables
from IPython.display import display


def section_to_rows(section: dict, metric_key: str) -> list[dict]:
    rows = []
    for mode_label, mode_key in [('rvOn', 'rvOn'), ('rvOff', 'rvOff')]:
        s = section[mode_key][metric_key]
        rows.append({
            'mode': mode_label,
            'count': s['count'],
            'mean': s['mean'],
            'stddev': s['stddev'],
            'median': s['median'],
            'min': s['min'],
            'max': s['max'],
            'p95': s['p95'],
            'p99': s['p99'],
        })
    return rows


try:
    import pandas as pd

    overhead_df = pd.DataFrame([
        {
            'dataset': 'baseline',
            'source': baseline['sourceCsv'],
            'overheadPct': baseline['overheadPct'],
        },
        {
            'dataset': 'abnormal',
            'source': abnormal['sourceCsv'],
            'overheadPct': abnormal['overheadPct'],
        },
    ])

    print('Overhead Summary')
    display(overhead_df)

    print('Baseline - elapsedMs')
    display(pd.DataFrame(section_to_rows(baseline, 'elapsedMs')))

    print('Baseline - rowsPerSec')
    display(pd.DataFrame(section_to_rows(baseline, 'rowsPerSec')))

    print('Abnormal - elapsedMs')
    display(pd.DataFrame(section_to_rows(abnormal, 'elapsedMs')))

    print('Abnormal - rowsPerSec')
    display(pd.DataFrame(section_to_rows(abnormal, 'rowsPerSec')))

    compare_csv = (
        runs_dir.parent / 'experiments' / 'larva-overhead-compare' / 'results' / 'comparison_summary.csv'
    )
    if compare_csv.exists():
        compare_df = pd.read_csv(compare_csv)
        compare_df['abnormal_gain_vs_original_pct'] = (
            (compare_df.loc[compare_df['variant'] == 'original_generated', 'abnormal_overhead_pct'].iloc[0]
             - compare_df['abnormal_overhead_pct'])
            / compare_df.loc[compare_df['variant'] == 'original_generated', 'abnormal_overhead_pct'].iloc[0]
            * 100.0
        )
        print('Three-variant comparison (original vs logging-disabled vs optimized)')
        display(compare_df)
    else:
        print(f'Comparison file not found: {compare_csv}')

except ImportError:
    # Fallback if pandas is unavailable
    print('pandas not installed; printing JSON-formatted tables.')
    print(json.dumps({'baseline': baseline, 'abnormal': abnormal}, indent=2))

Overhead Summary


,dataset,source,overheadPct
0,baseline,C:\Users\brama\Documents\RD2\src\resultCollect...,6.749556
1,abnormal,C:\Users\brama\Documents\RD2\src\resultCollect...,9.391304


Baseline - elapsedMs


,mode,count,mean,stddev,median,min,max,p95,p99
0,rvOn,5,120.2,14.822281,113.0,108.0,142.0,139.4,141.48
1,rvOff,5,112.6,14.976648,108.0,103.0,139.0,133.0,137.80


Baseline - rowsPerSec


...

,mode,count,mean,stddev,median,min,max,p95,p99
0,rvOn,5,125.8,9.80816,128.0,112.0,136.0,135.4,135.88
1,rvOff,5,115.0,10.29563,112.0,105.0,132.0,128.8,131.36


Abnormal - rowsPerSec


,mode,count,mean,stddev,median,min,max,p95,p99
0,rvOn,5,546726.879837,44060.617712,534632.812500,503183.823529,611008.928571,602862.142857,609379.571429
1,rvOff,5,598648.288177,50082.873668,611008.928571,518431.818182,651742.857143,645817.922078,650557.870130


Three-variant comparison (original vs logging-disabled vs optimized)


,variant,baseline_avg_on_ms,baseline_avg_off_ms,baseline_overhead_pct,baseline_avg_on_cpu_ms,baseline_avg_off_cpu_ms,baseline_avg_on_heap_delta_mb,baseline_avg_off_heap_delta_mb,abnormal_avg_on_ms,abnormal_avg_off_ms,abnormal_overhead_pct,abnormal_avg_on_cpu_ms,abnormal_avg_off_cpu_ms,abnormal_avg_on_heap_delta_mb,abnormal_avg_off_heap_delta_mb,abnormal_flags_total,session_dir,abnormal_gain_vs_original_pct
0,original_generated,125.8,118.2,6.429780,128.125,121.875,7.556180,3.509778,4824.4,119.8,3927.045075,3562.5,125.0,-30.590990,1.845613,68430,C:\Users\brama\Documents\RD2\src\resultCollect...,0.000000
1,logging_disabled_current,120.8,105.4,14.611006,118.750,109.375,12.020392,7.442126,851.2,108.6,683.793738,850.0,100.0,-10.623503,12.192766,68430,C:\Users\brama\Documents\RD2\src\resultCollect...,82.587576
2,optimized_no_stacktrace,120.2,112.6,6.749556,125.000,118.750,10.961868,7.560420,125.8,115.0,9.391304,125.0,112.5,13.970934,-26.134352,68430,C:\Users\brama\Documents\RD2\src\resultCollect...,99.760856


In [4]:
# Export JSON and TXT side-by-side with session outputs
json_out = session_dir / 'stats-summary.json'
txt_out = session_dir / 'stats-summary.txt'

json_out.write_text(json.dumps(report, indent=2), encoding='utf-8')

lines = ['Benchmark Statistics Summary', f'session={session_dir}', '']
for label, section in [('BASELINE', baseline), ('ABNORMAL', abnormal)]:
    lines.append(f'[{label}]')
    lines.append(f"source={section['sourceCsv']}")
    lines.append(f"overheadPct={section['overheadPct']:.4f}")
    lines.append(format_block('rvOn.elapsedMs', section['rvOn']['elapsedMs'], 'ms'))
    lines.append(format_block('rvOff.elapsedMs', section['rvOff']['elapsedMs'], 'ms'))
    lines.append(format_block('rvOn.rowsPerSec', section['rvOn']['rowsPerSec']))
    lines.append(format_block('rvOff.rowsPerSec', section['rvOff']['rowsPerSec']))
    lines.append('')

txt_out.write_text('\n'.join(lines), encoding='utf-8')

print(f'Wrote: {json_out}')
print(f'Wrote: {txt_out}')

Wrote: C:\Users\brama\Documents\RD2\src\resultCollection\runs\session-20260509-105518\stats-summary.json
Wrote: C:\Users\brama\Documents\RD2\src\resultCollection\runs\session-20260509-105518\stats-summary.txt
